# SPARK DATAFRAME API

- 대용량 데이터를 DataFrame 형태로 효율적으로 처리할 수 있도록 도와주는 고수준의 추상화 인터페이스
- 데이터를 Spark DataFrame 형태로 다룰 수 있도록 도와주는 spark 명령어
-  관련 내용 정리
     - [실무에서 자주 쓰는 Pyspark 명령어 정리](https://velog.io/@newnew_daddy/spark01)

In [1]:
# spark는 대용량 데이터 처리할 때 엄청난 장점이 있음
#메모리 기반 처리 방식
# 파티셔닝 -> 성능 향상에 목적이 있음
# transform 명령어는 순식간에 일어남 데이터의 양이 많아도
#action 명령어는 조금 느릴 수도 있음 용량이 늘단다면
# 열(column) 기반으로 데이터를 처리함.(디폴트는 parquet)

### 1. SPARK 환경 설정 & spark 세션 연결

In [78]:
!pip3 install pyspark

In [79]:
from pyspark.sql import SparkSession, Row
from pyspark.sql import types as T
from pyspark.sql import window as W
from pyspark.sql import functions as F

spark = SparkSession.builder \
        .master("local") \
        .appName("Colab") \
        .getOrCreate()

### 2. 데이터 읽기

In [4]:
!pwd

/content


In [6]:
df=spark.read.csv('./test1.csv',header=True)

df.show() #show는 디폴트가 20개임, head는 5개 인데반해

+-----+---+----------+------+
| Name|age|Experience|Salary|
+-----+---+----------+------+
|  Tom| 31|        10| 30000|
| Anne| 30|         8| 25000|
|Sunny| 29|         4| 20000|
| Paul| 24|         3| 20000|
| Mark| 21|         1| 15000|
|Jones| 23|         2| 18000|
+-----+---+----------+------+



In [7]:
spark.read.option("header","True").csv("./test1.csv").show()

+-----+---+----------+------+
| Name|age|Experience|Salary|
+-----+---+----------+------+
|  Tom| 31|        10| 30000|
| Anne| 30|         8| 25000|
|Sunny| 29|         4| 20000|
| Paul| 24|         3| 20000|
| Mark| 21|         1| 15000|
|Jones| 23|         2| 18000|
+-----+---+----------+------+



In [9]:
df.dtypes # 출력값이 list

[('Name', 'string'),
 ('age', 'string'),
 ('Experience', 'string'),
 ('Salary', 'string')]

In [10]:
df.printSchema() #출력값이 print(None) 확인 용도로만 사용

root
 |-- Name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- Experience: string (nullable = true)
 |-- Salary: string (nullable = true)



In [11]:
type(df)

pyspark.sql.dataframe.DataFrame

In [13]:
#show option
# df.show(3)
df.show(1, vertical=True) # 세로로 보는 경우(컬럼이 엄청 많아서 일반적인 show로른 한눈에 다 보이지 않을 때 사용)

-RECORD 0-----------
 Name       | Tom   
 age        | 31    
 Experience | 10    
 Salary     | 30000 
only showing top 1 row



In [15]:
df.show(1,vertical=True,truncate=False) # truncate 로우의 데이터를 자르지 않고 다 출력되도록 하는 옵션임(데이터가 길경우 짤리는 것 방지)

-RECORD 0-----------
 Name       | Tom   
 age        | 31    
 Experience | 10    
 Salary     | 30000 
only showing top 1 row



In [17]:
df.limit(3).show()

+-----+---+----------+------+
| Name|age|Experience|Salary|
+-----+---+----------+------+
|  Tom| 31|        10| 30000|
| Anne| 30|         8| 25000|
|Sunny| 29|         4| 20000|
+-----+---+----------+------+



In [18]:
#데이터의 길이 보기
df.count() #총 로우의 개수 확인 가능

6

In [20]:
# df.collect() #Row를 활용해서 전체적인 데이터 프레임 확인가능'
df.collect()[0] #인덱싱 가능

Row(Name='Tom', age='31', Experience='10', Salary='30000')

In [22]:
# df.first()
row1=df.first()
row1.Name, row1.age, row1.Experience, row1.Salary

('Tom', '31', '10', '30000')

In [23]:
#컬럼 리스트 출력
df.columns #리스트로 출력됨(컬럼명)

['Name', 'age', 'Experience', 'Salary']

### 3.컬럼 다루기 & DataFrame 생성

In [24]:
#Name, age 컬럼나 뽑아서 보기
#df.select('Name','age').show()
df.select(['Name','age']).show()

+-----+---+
| Name|age|
+-----+---+
|  Tom| 31|
| Anne| 30|
|Sunny| 29|
| Paul| 24|
| Mark| 21|
|Jones| 23|
+-----+---+



In [27]:
#특정컬럼 삭제 -drop
df_drop=df.drop('EXperience')

df_drop.show()

+-----+---+------+
| Name|age|Salary|
+-----+---+------+
|  Tom| 31| 30000|
| Anne| 30| 25000|
|Sunny| 29| 20000|
| Paul| 24| 20000|
| Mark| 21| 15000|
|Jones| 23| 18000|
+-----+---+------+



In [30]:
#컬럼 생성 - withColumn

df.withColumn("age_one",F.col('age')).show()

+-----+---+----------+------+-------+
| Name|age|Experience|Salary|age_one|
+-----+---+----------+------+-------+
|  Tom| 31|        10| 30000|     31|
| Anne| 30|         8| 25000|     30|
|Sunny| 29|         4| 20000|     29|
| Paul| 24|         3| 20000|     24|
| Mark| 21|         1| 15000|     21|
|Jones| 23|         2| 18000|     23|
+-----+---+----------+------+-------+



In [32]:
#컬럼 이름 바꾸기-withColumnRenamed
df.withColumnRenamed("Name","name123").show()

+-------+---+----------+------+
|name123|age|Experience|Salary|
+-------+---+----------+------+
|    Tom| 31|        10| 30000|
|   Anne| 30|         8| 25000|
|  Sunny| 29|         4| 20000|
|   Paul| 24|         3| 20000|
|   Mark| 21|         1| 15000|
|  Jones| 23|         2| 18000|
+-------+---+----------+------+



In [38]:
#DataFrame 생성

data=[
    Row("apple","banana","tomato",10),
    Row("apple2","banana2","tomato2",2),
    Row("apple3","banana3","tomato3",3)
]
schema = T.StructType(
    [
        T.StructField("col_A", T.StringType(),True),
        T.StructField("col_B", T.StringType(),True),
        T.StructField("col_C", T.StringType(),True),
        T.StructField("cost", T.IntegerType(),True)
    ]
)
#struck 안에 리스트로 한번 감싸고 StruckType은 아마 컬럼의 타입이랑 속성 정해주는 함수인듯
new_df1 = spark.createDataFrame(data,schema)
new_df1.show()

+------+-------+-------+----+
| col_A|  col_B|  col_C|cost|
+------+-------+-------+----+
| apple| banana| tomato|  10|
|apple2|banana2|tomato2|   2|
|apple3|banana3|tomato3|   3|
+------+-------+-------+----+



In [39]:
new_df1.printSchema()

root
 |-- col_A: string (nullable = true)
 |-- col_B: string (nullable = true)
 |-- col_C: string (nullable = true)
 |-- cost: integer (nullable = true)



In [40]:
new_df1.withColumn("double_cost",F.col("cost")*2).show()

+------+-------+-------+----+-----------+
| col_A|  col_B|  col_C|cost|double_cost|
+------+-------+-------+----+-----------+
| apple| banana| tomato|  10|         20|
|apple2|banana2|tomato2|   2|          4|
|apple3|banana3|tomato3|   3|          6|
+------+-------+-------+----+-----------+



### 4. Dataframe 읽기/쓰기 옵션들

##### 1) READ - 기본 옵션
  - PERMISSIVE(default) : 타입 이상이 있으면 null로 처리 후 read
  - DROPMALFORMED : 타입 이상 있는 row drop
  - FAILFAST : 타입 이상 있는 row 발견시 실행 실패

In [41]:
spark.read.option("mode","DROPMALFORMED").csv("./test1.csv") #내용의 타입이 안맞을 때 permissive사용

DataFrame[_c0: string, _c1: string, _c2: string, _c3: string]

##### 2) WRITE - 기본 옵션
  - error(default) : 파일 있으면 실행 실패
  - ignore : 파일 있으면 작업 안함
  - overwrite : 기존 경로 덮어쓰기
  - append : 기존 경로에 추가

In [44]:
#spark에서는 항상 디렉토리를 기준으로 write작업이 수행됨
df.write.mode("overwrite").csv("./temp1.csv") #폴더 이름으로 생성됨.

In [43]:
!pwd

/content


##### 3) WRITE - partition 옵션

In [46]:
#파티셔닝이 되지 않은 dataframe을 특정 column 기준으로 파티셔닝하여 저장하고자 할 때
tips = spark.read.csv("./tips.csv",header=True)
# tips.show()
tips.write.partitionBy("day").mode("overwrite").csv("./partition_tips")

In [ ]:
# partitonBy("day") # 파티션 될 컬럼의 로우수가 한쪽으로 몰려있어서 균등하지 않을 때 적은 값이 다른 값에 합해질 수 있음

### 5. 결측치 다루기

##### 1) 결측치 drop
- dropna()
- na.drop()

In [47]:
df = spark.read.csv("./test2.csv", header=True)
df.show()

+-----+----+----------+------+
| Name| age|Experience|Salary|
+-----+----+----------+------+
|  Tom|  31|        10| 30000|
| Anne|  30|         8| 25000|
|Sunny|  29|         4| 20000|
| Paul|  24|         3| 20000|
| Mark|  21|         1| 15000|
|Jones|  23|         2| 18000|
| Emma|NULL|      NULL| 40000|
| NULL|  34|        10| 38000|
| NULL|  36|      NULL|  NULL|
+-----+----+----------+------+



In [48]:
#dropna()
df.dropna().show()

+-----+---+----------+------+
| Name|age|Experience|Salary|
+-----+---+----------+------+
|  Tom| 31|        10| 30000|
| Anne| 30|         8| 25000|
|Sunny| 29|         4| 20000|
| Paul| 24|         3| 20000|
| Mark| 21|         1| 15000|
|Jones| 23|         2| 18000|
+-----+---+----------+------+



In [49]:
#dropna() - thresh 조건을 줄 수 있음 > 정상값이 thresh보다 적게 있는 row를 drop하는 것
df.dropna(thresh=2).show() #정상값이 2보다 적게 있는것을 drop(이하 개념이 아니라 미만 개념)

+-----+----+----------+------+
| Name| age|Experience|Salary|
+-----+----+----------+------+
|  Tom|  31|        10| 30000|
| Anne|  30|         8| 25000|
|Sunny|  29|         4| 20000|
| Paul|  24|         3| 20000|
| Mark|  21|         1| 15000|
|Jones|  23|         2| 18000|
| Emma|NULL|      NULL| 40000|
| NULL|  34|        10| 38000|
+-----+----+----------+------+



In [50]:
#na.drop()

df.na.drop().show()

+-----+---+----------+------+
| Name|age|Experience|Salary|
+-----+---+----------+------+
|  Tom| 31|        10| 30000|
| Anne| 30|         8| 25000|
|Sunny| 29|         4| 20000|
| Paul| 24|         3| 20000|
| Mark| 21|         1| 15000|
|Jones| 23|         2| 18000|
+-----+---+----------+------+



In [52]:
#how 옵션에서 any: null 하나라도 있으면 drop
#-all :행 전체가 null 값이면 drop
df.na.drop(how="any").show()

+-----+---+----------+------+
| Name|age|Experience|Salary|
+-----+---+----------+------+
|  Tom| 31|        10| 30000|
| Anne| 30|         8| 25000|
|Sunny| 29|         4| 20000|
| Paul| 24|         3| 20000|
| Mark| 21|         1| 15000|
|Jones| 23|         2| 18000|
+-----+---+----------+------+



In [55]:
#특정 컬럼에 대한 null 값만 drop
# df.na.drop(subset =['age']).show()
# df.na.drop(how="all",subset=['age','Experience']).show()
df.na.drop(how="any",subset=['age','Experience']).show()

+-----+---+----------+------+
| Name|age|Experience|Salary|
+-----+---+----------+------+
|  Tom| 31|        10| 30000|
| Anne| 30|         8| 25000|
|Sunny| 29|         4| 20000|
| Paul| 24|         3| 20000|
| Mark| 21|         1| 15000|
|Jones| 23|         2| 18000|
| NULL| 34|        10| 38000|
+-----+---+----------+------+



##### 2) 결측치 fill
- fillna()
- na.fill()

In [56]:
df.fillna("KDT7").show()

+-----+----+----------+------+
| Name| age|Experience|Salary|
+-----+----+----------+------+
|  Tom|  31|        10| 30000|
| Anne|  30|         8| 25000|
|Sunny|  29|         4| 20000|
| Paul|  24|         3| 20000|
| Mark|  21|         1| 15000|
|Jones|  23|         2| 18000|
| Emma|KDT7|      KDT7| 40000|
| KDT7|  34|        10| 38000|
| KDT7|  36|      KDT7|  KDT7|
+-----+----+----------+------+



In [57]:
#특정 컬럼에 대해서 fill na -> 컬럼의 타입과 채우려는 타입이 동일해야 함. 안그러면 변화가 없음
df.fillna("KDT7",subset=["name","Salary"]).show()


+-----+----+----------+------+
| Name| age|Experience|Salary|
+-----+----+----------+------+
|  Tom|  31|        10| 30000|
| Anne|  30|         8| 25000|
|Sunny|  29|         4| 20000|
| Paul|  24|         3| 20000|
| Mark|  21|         1| 15000|
|Jones|  23|         2| 18000|
| Emma|NULL|      NULL| 40000|
| KDT7|  34|        10| 38000|
| KDT7|  36|      NULL|  KDT7|
+-----+----+----------+------+



In [58]:
df.na.fill("KDT7",subset=["name","Salary"]).show()

+-----+----+----------+------+
| Name| age|Experience|Salary|
+-----+----+----------+------+
|  Tom|  31|        10| 30000|
| Anne|  30|         8| 25000|
|Sunny|  29|         4| 20000|
| Paul|  24|         3| 20000|
| Mark|  21|         1| 15000|
|Jones|  23|         2| 18000|
| Emma|NULL|      NULL| 40000|
| KDT7|  34|        10| 38000|
| KDT7|  36|      NULL|  KDT7|
+-----+----+----------+------+



In [ ]:
#동일한 기능을 하지만 다른 이름을 가진 메소드
#fillna, na.fill
#filter, where
#oderBy, sort
#drop_duplicate, dropdDuplicate

### 6. filter

##### 1) 단일 & 다중 filter

In [59]:
#sql에서 where조건과 비슷한 기능
item=spark.read.parquet("./item_his.parquet")
item.show()

+-----+-----+-------+--------+----------+--------------+-----+
|  idx|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|
+-----+-----+-------+--------+----------+--------------+-----+
|53687|175.0| 202305|20230508|  액세서리|아바타파츠구분|   50|
|53687|175.0| 202305|20230508|상태메시지|  기타파츠구분|   50|
|20163|161.0| 202304|20230408|  액세서리|아바타파츠구분|   50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|
|26112|130.0| 202304|20230405|  액세서리|아바타파츠구분|   50|
|26112|130.0| 202304|20230405|      헤어|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|      얼굴|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|      신발|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|상태메시지|  기타파츠구분|   50|
|49541|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|
|37822| 39.0| 202306|20230608|      헤어|아바타파츠구분|  150|
|37822| 39.0| 202306|20230608|상태메시지|  기타파츠구분|   50|
|37822| 39.0| 202306|20230608|상태메시지|  기타파츠구분|   50|
|76257|190.0| 20230

In [63]:
item.filter(F.col("codename") == "액세서리").show()
# item.filter(item.codename =="액세서리").show(5) # 이렇게도 쓰지만 F.col 쓰는걸 추천
# 한글 혹은 띄어쓰기가 들어있는 컬럼 ->"학생번호","등록 날짜" 일 경우 item.codename사용 불가

+-----+-----+-------+--------+--------+--------------+-----+
|  idx|   lv|proc_ym|proc_ymd|codename|   mascodename|price|
+-----+-----+-------+--------+--------+--------------+-----+
|53687|175.0| 202305|20230508|액세서리|아바타파츠구분|   50|
|20163|161.0| 202304|20230408|액세서리|아바타파츠구분|   50|
|26112|130.0| 202304|20230405|액세서리|아바타파츠구분|   50|
|76257|190.0| 202306|20230608|액세서리|아바타파츠구분|   25|
|76257|190.0| 202306|20230608|액세서리|아바타파츠구분|   25|
+-----+-----+-------+--------+--------+--------------+-----+
only showing top 5 rows



In [66]:
# 특정 컬럼에 대해 Type 변환, SQL CAST 떠올려보자
item.withColumn("price_int", F.col("price").cast(T.IntegerType())).show()
# item.withColumn("price_int", F.col("price").cast(T.IntegerType())).printSchema()
# item = item.withColumn("price_int", F.col("price").cast(T.IntegerType()))

+-----+-----+-------+--------+----------+--------------+-----+---------+
|  idx|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|price_int|
+-----+-----+-------+--------+----------+--------------+-----+---------+
|53687|175.0| 202305|20230508|  액세서리|아바타파츠구분|   50|       50|
|53687|175.0| 202305|20230508|상태메시지|  기타파츠구분|   50|       50|
|20163|161.0| 202304|20230408|  액세서리|아바타파츠구분|   50|       50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|       50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|       50|
|26112|130.0| 202304|20230405|  액세서리|아바타파츠구분|   50|       50|
|26112|130.0| 202304|20230405|      헤어|아바타파츠구분|  150|      150|
|49541|170.0| 202306|20230630|      얼굴|아바타파츠구분|  150|      150|
|49541|170.0| 202306|20230630|      신발|아바타파츠구분|  150|      150|
|49541|170.0| 202306|20230630|상태메시지|  기타파츠구분|   50|       50|
|49541|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|      150|
|49541|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|      150|
|37822| 39.0| 202306|202306

In [68]:
# price 컬럼이 100 이상인 row만 추출

item.filter(F.col("price") >= 100).show(3)

+-----+-----+-------+--------+--------+--------------+-----+---------+
|  idx|   lv|proc_ym|proc_ymd|codename|   mascodename|price|price_int|
+-----+-----+-------+--------+--------+--------------+-----+---------+
|26112|130.0| 202304|20230405|    헤어|아바타파츠구분|  150|      150|
|49541|170.0| 202306|20230630|    얼굴|아바타파츠구분|  150|      150|
|49541|170.0| 202306|20230630|    신발|아바타파츠구분|  150|      150|
+-----+-----+-------+--------+--------+--------------+-----+---------+
only showing top 3 rows



In [70]:
## 다중 filter 조건

item.filter(F.col("codename") == "액세서리").filter(F.col("price")<50).show()# 잘 안씀 오히려 아래코드가 더 자주 쓰는 듯

+-----+-----+-------+--------+--------+--------------+-----+---------+
|  idx|   lv|proc_ym|proc_ymd|codename|   mascodename|price|price_int|
+-----+-----+-------+--------+--------+--------------+-----+---------+
|76257|190.0| 202306|20230608|액세서리|아바타파츠구분|   25|       25|
|76257|190.0| 202306|20230608|액세서리|아바타파츠구분|   25|       25|
|32960|151.0| 202305|20230510|액세서리|아바타파츠구분|   25|       25|
|61823|151.0| 202304|20230418|액세서리|아바타파츠구분|   25|       25|
|77389|118.0| 202304|20230421|액세서리|아바타파츠구분|   25|       25|
|13456|158.0| 202304|20230421|액세서리|아바타파츠구분|   25|       25|
|74784|156.0| 202306|20230619|액세서리|아바타파츠구분|   25|       25|
|86844|152.0| 202306|20230609|액세서리|아바타파츠구분|   25|       25|
|52543|152.0| 202304|20230424|액세서리|아바타파츠구분|   25|       25|
|43710|174.0| 202305|20230520|액세서리|아바타파츠구분|   25|       25|
|13841|170.0| 202304|20230418|액세서리|아바타파츠구분|   25|       25|
|13841|170.0| 202304|20230418|액세서리|아바타파츠구분|   25|       25|
|13841|170.0| 202304|20230418|액세서리|아바타파츠구분|   25|       25|
|13841|

In [71]:
# df.filter((조건1)& (조건2)| (조건3)....)
item.filter((F.col("codename") == "액세서리") | (F.col("price")<50)).show()

+-----+-----+-------+--------+--------+--------------+-----+---------+
|  idx|   lv|proc_ym|proc_ymd|codename|   mascodename|price|price_int|
+-----+-----+-------+--------+--------+--------------+-----+---------+
|53687|175.0| 202305|20230508|액세서리|아바타파츠구분|   50|       50|
|20163|161.0| 202304|20230408|액세서리|아바타파츠구분|   50|       50|
|26112|130.0| 202304|20230405|액세서리|아바타파츠구분|   50|       50|
|76257|190.0| 202306|20230608|액세서리|아바타파츠구분|   25|       25|
|76257|190.0| 202306|20230608|액세서리|아바타파츠구분|   25|       25|
|67188|175.0| 202306|20230612|액세서리|아바타파츠구분|   50|       50|
|11987|159.0| 202304|20230404|액세서리|아바타파츠구분|   50|       50|
|48378|144.0| 202304|20230412|액세서리|아바타파츠구분|   50|       50|
|32960|151.0| 202305|20230510|액세서리|아바타파츠구분|   25|       25|
|61823|151.0| 202304|20230418|액세서리|아바타파츠구분|   25|       25|
|84515| 83.0| 202304|20230425|액세서리|아바타파츠구분|   50|       50|
|53306|130.0| 202304|20230410|액세서리|아바타파츠구분|   50|       50|
|77389|118.0| 202304|20230421|액세서리|아바타파츠구분|   25|       25|
|13456|

In [72]:
# filter에 대한 반대조건을 추출
#예를 들어 codename이 액세서리가 아닌 것을 추출

# item.filter(F.col("codename" != "액세서리").show()
item.filter(~(F.col("codename")=="액세서리")).show()

+-----+-----+-------+--------+----------+--------------+-----+---------+
|  idx|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|price_int|
+-----+-----+-------+--------+----------+--------------+-----+---------+
|53687|175.0| 202305|20230508|상태메시지|  기타파츠구분|   50|       50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|       50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|       50|
|26112|130.0| 202304|20230405|      헤어|아바타파츠구분|  150|      150|
|49541|170.0| 202306|20230630|      얼굴|아바타파츠구분|  150|      150|
|49541|170.0| 202306|20230630|      신발|아바타파츠구분|  150|      150|
|49541|170.0| 202306|20230630|상태메시지|  기타파츠구분|   50|       50|
|49541|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|      150|
|49541|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|      150|
|37822| 39.0| 202306|20230608|      헤어|아바타파츠구분|  150|      150|
|37822| 39.0| 202306|20230608|상태메시지|  기타파츠구분|   50|       50|
|37822| 39.0| 202306|20230608|상태메시지|  기타파츠구분|   50|       50|
|84801|112.0| 202306|2023

##### 2) 중복 제거

In [73]:
# distinct() -> 항상 모든 컬럼에 대해 중복제거, 줄수 있는 옵션이 없음
item.distinct().show()


+-----+-----+-------+--------+----------+--------------+-----+---------+
|  idx|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|price_int|
+-----+-----+-------+--------+----------+--------------+-----+---------+
|13456|158.0| 202304|20230421|  액세서리|아바타파츠구분|   25|       25|
|10567|138.0| 202304|20230406|    코스튬|아바타파츠구분|  300|      300|
|49340|172.0| 202305|20230519|      신발|아바타파츠구분|   60|       60|
|42041|143.0| 202304|20230412|      신발|아바타파츠구분|   60|       60|
|74758|136.0| 202304|20230425|      신발|아바타파츠구분|   60|       60|
|67728|137.0| 202305|20230504|  액세서리|아바타파츠구분|   50|       50|
|51659|156.0| 202304|20230410|      하의|아바타파츠구분|  250|      250|
|64502|196.0| 202305|20230531|  액세서리|아바타파츠구분|   50|       50|
|32199|197.0| 202306|20230612|      얼굴|아바타파츠구분|  150|      150|
|24463|197.0| 202306|20230625|      얼굴|아바타파츠구분|  150|      150|
| 1150|160.0| 202304|20230422|      얼굴|아바타파츠구분|  150|      150|
|31885|168.0| 202305|20230512|      상의|아바타파츠구분|  200|      200|
|33817|159.0| 202306

In [76]:
#특정 컬럼을 대상으로 중복제거 -> dropDuplicates()/drop_duplicates()
# item.dropDuplicates().show() #모든 컬럼을 대상으로(디폴트)
item.dropDuplicates(subset=["proc_ym","codename"]).show()

+-----+-----+-------+--------+----------+--------------+-----+---------+
|  idx|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|price_int|
+-----+-----+-------+--------+----------+--------------+-----+---------+
|84515| 83.0| 202304|20230425|      상의|아바타파츠구분|  200|      200|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|       50|
|31799| 81.0| 202304|20230406|      신발|아바타파츠구분|   60|       60|
|20163|161.0| 202304|20230408|  액세서리|아바타파츠구분|   50|       50|
|46516|168.0| 202304|20230424|      얼굴|아바타파츠구분|  150|      150|
|87383| 39.0| 202304|20230418|    코스튬|아바타파츠구분|  300|      300|
|29907|158.0| 202304|20230404|      하의|아바타파츠구분|  250|      250|
|26112|130.0| 202304|20230405|      헤어|아바타파츠구분|  150|      150|
|52108|159.0| 202305|20230515|      상의|아바타파츠구분|  200|      200|
|53687|175.0| 202305|20230508|상태메시지|  기타파츠구분|   50|       50|
|34355|165.0| 202305|20230505|      신발|아바타파츠구분|   60|       60|
|53687|175.0| 202305|20230508|  액세서리|아바타파츠구분|   50|       50|
|88354| 82.0| 202305|2

##### 3) 포함 여부 확인(isin, like), Null 값 확인

In [77]:
# isin() -> 특정 리스트의 값이 해당 컬럼에 존재하는지 여부를 확인

codename_list = ['상의','신발','얼굴']
item.filter(F.col('codename').isin(codename_list)).show()

+-----+-----+-------+--------+--------+--------------+-----+---------+
|  idx|   lv|proc_ym|proc_ymd|codename|   mascodename|price|price_int|
+-----+-----+-------+--------+--------+--------------+-----+---------+
|49541|170.0| 202306|20230630|    얼굴|아바타파츠구분|  150|      150|
|49541|170.0| 202306|20230630|    신발|아바타파츠구분|  150|      150|
|84801|112.0| 202306|20230627|    얼굴|아바타파츠구분|  150|      150|
|67188|175.0| 202306|20230612|    얼굴|아바타파츠구분|  150|      150|
|67188|175.0| 202306|20230612|    상의|아바타파츠구분|  200|      200|
|67188|175.0| 202306|20230612|    신발|아바타파츠구분|   60|       60|
| 3663|192.0| 202306|20230612|    얼굴|아바타파츠구분|  150|      150|
|34355|165.0| 202305|20230505|    신발|아바타파츠구분|   60|       60|
|31799| 81.0| 202304|20230406|    신발|아바타파츠구분|   60|       60|
|83403|109.0| 202304|20230419|    신발|아바타파츠구분|   60|       60|
|30959|137.0| 202306|20230613|    신발|아바타파츠구분|   60|       60|
|76038|159.0| 202306|20230613|    상의|아바타파츠구분|  200|      200|
|25146|159.0| 202305|20230515|    신발|아바타파츠구

In [ ]:
# item = spark.read.parquet("./item_his_parquet")
# iteem=item.withColumn("price" ,F.col)

In [84]:
item.filter(F.col("mascodename").like("아바타%")).show(3)

+-----+-----+-------+--------+--------+--------------+-----+---------+
|  idx|   lv|proc_ym|proc_ymd|codename|   mascodename|price|price_int|
+-----+-----+-------+--------+--------+--------------+-----+---------+
|53687|175.0| 202305|20230508|액세서리|아바타파츠구분|   50|       50|
|20163|161.0| 202304|20230408|액세서리|아바타파츠구분|   50|       50|
|26112|130.0| 202304|20230405|액세서리|아바타파츠구분|   50|       50|
+-----+-----+-------+--------+--------+--------------+-----+---------+
only showing top 3 rows



In [87]:
#null 값 유무에 대한 필터링
test2 = spark.read.csv( "./test2.csv", header=True)


In [96]:
test2.filter(F.col("Name").isNotNull()).show(5)

+-----+---+----------+------+
| Name|age|Experience|Salary|
+-----+---+----------+------+
|  Tom| 31|        10| 30000|
| Anne| 30|         8| 25000|
|Sunny| 29|         4| 20000|
| Paul| 24|         3| 20000|
| Mark| 21|         1| 15000|
+-----+---+----------+------+
only showing top 5 rows



##### 4) When, Between

In [94]:
#Between -> 컬럼의 데이터가 특정 값 사이에 있는지 판단(양 끝값 포함)
item.filter(F.col("price").between(51,100)).show() # 51 <= ... <=100

+-----+-----+-------+--------+----------+--------------+-----+---------+
|  idx|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|price_int|
+-----+-----+-------+--------+----------+--------------+-----+---------+
|67188|175.0| 202306|20230612|      신발|아바타파츠구분|   60|       60|
|34355|165.0| 202305|20230505|      신발|아바타파츠구분|   60|       60|
|31799| 81.0| 202304|20230406|      신발|아바타파츠구분|   60|       60|
|83403|109.0| 202304|20230419|      신발|아바타파츠구분|   60|       60|
|30959|137.0| 202306|20230613|      신발|아바타파츠구분|   60|       60|
|11987|159.0| 202304|20230404|      신발|아바타파츠구분|   60|       60|
|11987|159.0| 202304|20230404|      신발|아바타파츠구분|   60|       60|
|56283|114.0| 202305|20230527|      신발|아바타파츠구분|   60|       60|
| 7632|147.0| 202304|20230420|      신발|아바타파츠구분|   60|       60|
|75132|133.0| 202304|20230412|      신발|아바타파츠구분|   60|       60|
|81788| 15.0| 202304|20230420|상태메시지|  기타파츠구분|  100|      100|
|30445|160.0| 202305|20230530|상태메시지|  기타파츠구분|  100|      100|
|30445|160.0| 202

In [100]:
#when 조건문 -> 특정 컬럼이 어떤 조건을 만족할 때 수행할 내용을 선언함.
'''
test2 테이블
-만약(when) age가 null이면 "30"으로 채워넣도록 한다. 아니라면(otherwise) 그대로 둔다.
F.when("조건", "value")
'''
test2.withColumn("age_when",F.when(F.col("age").isNull(),"KDT7").otherwise(F.col("age"))).show() #새로운 컬럼 만듦
# test2.withColumn("age",F.when(F.col("age").isNull(),"KDT7").otherwise(F.col("age"))).show()#이렇게 하면 덮어씀

+-----+----+----------+------+--------+
| Name| age|Experience|Salary|age_when|
+-----+----+----------+------+--------+
|  Tom|  31|        10| 30000|      31|
| Anne|  30|         8| 25000|      30|
|Sunny|  29|         4| 20000|      29|
| Paul|  24|         3| 20000|      24|
| Mark|  21|         1| 15000|      21|
|Jones|  23|         2| 18000|      23|
| Emma|NULL|      NULL| 40000|    KDT7|
| NULL|  34|        10| 38000|      34|
| NULL|  36|      NULL|  NULL|      36|
+-----+----+----------+------+--------+



### 7. group by + aggregation

##### 1) 단일 Group By

In [101]:
item.show()

+-----+-----+-------+--------+----------+--------------+-----+---------+
|  idx|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|price_int|
+-----+-----+-------+--------+----------+--------------+-----+---------+
|53687|175.0| 202305|20230508|  액세서리|아바타파츠구분|   50|       50|
|53687|175.0| 202305|20230508|상태메시지|  기타파츠구분|   50|       50|
|20163|161.0| 202304|20230408|  액세서리|아바타파츠구분|   50|       50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|       50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|       50|
|26112|130.0| 202304|20230405|  액세서리|아바타파츠구분|   50|       50|
|26112|130.0| 202304|20230405|      헤어|아바타파츠구분|  150|      150|
|49541|170.0| 202306|20230630|      얼굴|아바타파츠구분|  150|      150|
|49541|170.0| 202306|20230630|      신발|아바타파츠구분|  150|      150|
|49541|170.0| 202306|20230630|상태메시지|  기타파츠구분|   50|       50|
|49541|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|      150|
|49541|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|      150|
|37822| 39.0| 202306|202306

In [105]:
# item.proc_ym 컬럼에 대한 group by count
# item.groupBy(F.col("proc_ym")).count().show()
# item.groupBy("proc_ym").count().show() #위에랑 같은 결과값 반환 이걸 더 많이 쓰는듯. groupby에서는
#count는 종종 쓰지만 max.min은 잘 안쓰는듯 하다고 함
# item.groupBy("proc_ym").max().show()
# item.groupBy("proc_ym").min().show()
item.groupBy("proc_ym").mean().show()


+-------+------------------+
|proc_ym|    avg(price_int)|
+-------+------------------+
| 202304|132.73599837203517|
| 202306| 133.3655146115714|
| 202305|131.17641413725974|
+-------+------------------+



In [109]:
#groupby -agrregation
#groupby.agg(집계함수(집계대상컬럼))
#groupby.agg(집계함수(집계대상컬럼).alias())-> 새로운 컬럼 생성하고 집계함수 적용
#인티저만 알아서 계산해줌
item.groupby("proc_ym").agg(F.count(F.col("codename")).alias("codename_cnt")).show()

+-------+------------+
|proc_ym|codename_cnt|
+-------+------------+
| 202304|       44227|
| 202306|       28847|
| 202305|       36471|
+-------+------------+



In [114]:
item.groupby("proc_ym").agg(F.max(F.col("price")).alias("price_max")).show() #idx가 string으로 되어있어서 값이 정확하게 안나옴
#값이 나오긴함. 숫자형일 때만 사용해야됨.

+-------+---------+
|proc_ym|price_max|
+-------+---------+
| 202304|       60|
| 202305|       60|
| 202306|       60|
+-------+---------+



In [116]:
#aggregation 컬럼 여러개
item.groupby("proc_ym").agg(
    F.count(F.col("price")).alias("cnt"),
    F.max(F.col("price")).alias("max"),
    F.min(F.col("price").alias("price")).alias("min"),
    F.mean(F.col("price").alias("price")).alias("mean"),
    F.round( F.mean(F.col("price")),2).alias("mean_round")
).show()

+-------+-----+---+---+------------------+----------+
|proc_ym|  cnt|max|min|              mean|mean_round|
+-------+-----+---+---+------------------+----------+
| 202304|44227| 60|100|132.73599837203517|    132.74|
| 202305|36471| 60|100|131.17641413725974|    131.18|
| 202306|28847| 60|100| 133.3655146115714|    133.37|
+-------+-----+---+---+------------------+----------+



In [118]:
#collect_list -> groupby 후 agg대상 컬럼들에 대한 리스트 값
item.groupby("proc_ym").agg(
    F.collect_list(F.col("codename")).alias("code_list")
).show()

+-------+--------------------------------+
|proc_ym|                       code_list|
+-------+--------------------------------+
| 202304|[액세서리, 상태메시지, 상태메...|
| 202306|  [얼굴, 신발, 상태메시지, 헤...|
| 202305|[액세서리, 상태메시지, 상태메...|
+-------+--------------------------------+



In [120]:
#collect_set -> groupby 후 agg대상 컬럼들에 중복 제거 후의 리스트 값
item.groupby("proc_ym").agg(
    F.collect_set(F.col("codename")).alias("code_list")
).show(truncate=False)

+-------+------------------------------------------------------------+
|proc_ym|code_list                                                   |
+-------+------------------------------------------------------------+
|202304 |[하의, 신발, 코스튬, 상의, 헤어, 액세서리, 얼굴, 상태메시지]|
|202306 |[하의, 신발, 코스튬, 상의, 헤어, 액세서리, 얼굴, 상태메시지]|
|202305 |[하의, 신발, 코스튬, 상의, 헤어, 액세서리, 얼굴, 상태메시지]|
+-------+------------------------------------------------------------+



In [123]:
# sum.countDistinct
item.groupby("proc_ym").agg(
    F.sum(F.col("price")).alias("price_sum"),
    F.count(F.col("proc_ymd")).alias("cnt"),
    F.countDistinct(F.col("proc_ymd")).alias("ymd_cnt")
    ).show()

+-------+---------+-----+-------+
|proc_ym|price_sum|  cnt|ymd_cnt|
+-------+---------+-----+-------+
| 202304|5870515.0|44227|     30|
| 202306|3847195.0|28847|     30|
| 202305|4784135.0|36471|     31|
+-------+---------+-----+-------+



##### 2) 다중 Group By

In [128]:
# item.groupby("proc_ym","mascodename","codename").count().show()
item.groupby("proc_ym","mascodename","codename").agg(F.count("price")).show(5)#왼쪽에서 오른쪽으로 갈수록 단위가 작아지면서 오른쪽 컬럼에 대한 그룹바이가 실행됨.

+-------+--------------+--------+------------+
|proc_ym|   mascodename|codename|count(price)|
+-------+--------------+--------+------------+
| 202306|아바타파츠구분|액세서리|        3059|
| 202305|아바타파츠구분|    신발|        6004|
| 202306|아바타파츠구분|    신발|        3843|
| 202304|아바타파츠구분|  코스튬|        2317|
| 202306|아바타파츠구분|    상의|        3443|
+-------+--------------+--------+------------+
only showing top 5 rows



### 8. order by

##### 1) 단일 Order By

In [129]:
from typing import cast
#오름차순 내림차순 정의 인티저에 대하서만 연산되기 때문에 바꿔줘야 함
#idx, lv 컬럼을 IntegerType()으로 변환
item.withColumn("idx",F.col("idx").cast(T.IntegerType())).withColumn("lv",F.col("lv").cast)

root
 |-- idx: string (nullable = true)
 |-- lv: string (nullable = true)
 |-- proc_ym: string (nullable = true)
 |-- proc_ymd: string (nullable = true)
 |-- codename: string (nullable = true)
 |-- mascodename: string (nullable = true)
 |-- price: string (nullable = true)
 |-- price_int: integer (nullable = true)



In [131]:
#orderBy의 디폴트는 오름차순 정렬
item.orderBy(F.col("idx")).show()# item.orderBy("idx")

+-----+-----+-------+--------+--------+--------------+-----+---------+
|  idx|   lv|proc_ym|proc_ymd|codename|   mascodename|price|price_int|
+-----+-----+-------+--------+--------+--------------+-----+---------+
|10005|150.0| 202304|20230403|    헤어|아바타파츠구분|  150|      150|
|10005|150.0| 202304|20230403|    헤어|아바타파츠구분|  150|      150|
|10005|163.0| 202305|20230509|    신발|아바타파츠구분|   60|       60|
|10005|171.0| 202305|20230524|    신발|아바타파츠구분|   60|       60|
|10009|159.0| 202304|20230414|    헤어|아바타파츠구분|  150|      150|
|10009|159.0| 202304|20230414|액세서리|아바타파츠구분|   50|       50|
|10009|159.0| 202304|20230414|    신발|아바타파츠구분|   60|       60|
|10009|174.0| 202306|20230614|    헤어|아바타파츠구분|  150|      150|
|10009|174.0| 202306|20230614|    신발|아바타파츠구분|   60|       60|
|10009|171.0| 202305|20230530|    헤어|아바타파츠구분|  150|      150|
|10009|171.0| 202305|20230530|액세서리|아바타파츠구분|   50|       50|
|10009|171.0| 202305|20230530|    신발|아바타파츠구분|   60|       60|
|10009|171.0| 202305|20230530|액세서리|아바타파츠구분|   5

In [132]:
# 내림차순 정렬(descending) 하는 방법들

item.orderBy(F.col("idx").desc()).show(3)

+----+----+-------+--------+--------+--------------+-----+---------+
| idx|  lv|proc_ym|proc_ymd|codename|   mascodename|price|price_int|
+----+----+-------+--------+--------+--------------+-----+---------+
|9995|80.0| 202305|20230516|액세서리|아바타파츠구분|   50|       50|
|9995|80.0| 202305|20230516|    헤어|아바타파츠구분|  150|      150|
|9995|80.0| 202305|20230516|    신발|아바타파츠구분|   60|       60|
+----+----+-------+--------+--------+--------------+-----+---------+
only showing top 3 rows



In [133]:
item.orderBy("idx",ascending=False).show() #위에 코드랑 값이 같은 것인데 형식이 조금 다름

+----+-----+-------+--------+----------+--------------+-----+---------+
| idx|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|price_int|
+----+-----+-------+--------+----------+--------------+-----+---------+
|9995| 80.0| 202305|20230516|      얼굴|아바타파츠구분|  150|      150|
|9995| 80.0| 202305|20230516|      신발|아바타파츠구분|   60|       60|
|9995| 80.0| 202305|20230516|      헤어|아바타파츠구분|  150|      150|
|9995| 80.0| 202305|20230516|  액세서리|아바타파츠구분|   50|       50|
|9995| 88.0| 202306|20230610|      얼굴|아바타파츠구분|  150|      150|
|9995| 88.0| 202306|20230610|      상의|아바타파츠구분|  200|      200|
|9994|180.0| 202305|20230517|    코스튬|아바타파츠구분|  300|      300|
|9993|199.0| 202305|20230531|상태메시지|  기타파츠구분|   50|       50|
|9991|179.0| 202304|20230420|      하의|아바타파츠구분|  250|      250|
|9991|179.0| 202304|20230420|      헤어|아바타파츠구분|  150|      150|
|9991|181.0| 202304|20230422|      하의|아바타파츠구분|  250|      250|
|9991|196.0| 202305|20230521|      상의|아바타파츠구분|  200|      200|
|9991|196.0| 202305|20230521|    

In [136]:
#orderBy 시 Null값이 있으면 null first가 default
'''
asc_nulls_last() ->내림차순 null 마지막
asc_nulls_first() -> 내림차순 null 먼저
desc_nulls_last() -> 오름차순 null 마지막
desc_nulls_first() -> 오름차순 null 먼저
'''
item.orderBy(F.col("lv").asc_nulls_last()).show()

+-----+---+-------+--------+--------+--------------+-----+---------+
|  idx| lv|proc_ym|proc_ymd|codename|   mascodename|price|price_int|
+-----+---+-------+--------+--------+--------------+-----+---------+
|89357|1.0| 202304|20230413|  코스튬|아바타파츠구분|  300|      300|
|96882|1.0| 202306|20230628|  코스튬|아바타파츠구분|  300|      300|
|89357|1.0| 202304|20230413|  코스튬|아바타파츠구분|  300|      300|
|94021|1.0| 202305|20230526|  코스튬|아바타파츠구분|  300|      300|
|89357|1.0| 202304|20230413|  코스튬|아바타파츠구분|  300|      300|
|89969|1.0| 202304|20230418|  코스튬|아바타파츠구분|  300|      300|
|94238|1.0| 202305|20230529|  코스튬|아바타파츠구분|  300|      300|
|94389|1.0| 202305|20230531|  코스튬|아바타파츠구분|  300|      300|
|94238|1.0| 202305|20230529|  코스튬|아바타파츠구분|  300|      300|
|94389|1.0| 202305|20230531|  코스튬|아바타파츠구분|  300|      300|
|89519|1.0| 202304|20230415|  코스튬|아바타파츠구분|  300|      300|
|94389|1.0| 202305|20230531|  코스튬|아바타파츠구분|  300|      300|
|89519|1.0| 202304|20230415|  코스튬|아바타파츠구분|  300|      300|
|94389|1.0| 202305|2023053

In [139]:
item.sort(F.col("lv").asc_nulls_last()).show()

+-----+---+-------+--------+--------+--------------+-----+---------+
|  idx| lv|proc_ym|proc_ymd|codename|   mascodename|price|price_int|
+-----+---+-------+--------+--------+--------------+-----+---------+
|89357|1.0| 202304|20230413|  코스튬|아바타파츠구분|  300|      300|
|96882|1.0| 202306|20230628|  코스튬|아바타파츠구분|  300|      300|
|89357|1.0| 202304|20230413|  코스튬|아바타파츠구분|  300|      300|
|94021|1.0| 202305|20230526|  코스튬|아바타파츠구분|  300|      300|
|89357|1.0| 202304|20230413|  코스튬|아바타파츠구분|  300|      300|
|89969|1.0| 202304|20230418|  코스튬|아바타파츠구분|  300|      300|
|94238|1.0| 202305|20230529|  코스튬|아바타파츠구분|  300|      300|
|94389|1.0| 202305|20230531|  코스튬|아바타파츠구분|  300|      300|
|94238|1.0| 202305|20230529|  코스튬|아바타파츠구분|  300|      300|
|94389|1.0| 202305|20230531|  코스튬|아바타파츠구분|  300|      300|
|89519|1.0| 202304|20230415|  코스튬|아바타파츠구분|  300|      300|
|94389|1.0| 202305|20230531|  코스튬|아바타파츠구분|  300|      300|
|89519|1.0| 202304|20230415|  코스튬|아바타파츠구분|  300|      300|
|94389|1.0| 202305|2023053

##### 2) 다중 Order By

In [143]:
# 정렬 순서를 컬럼마다 다르게 할 때 다중 orderby
#proc_ym -> 오름차순 , proc_ymd -> 내림차순
item.dropDuplicates(subset=["proc_ym","proc_ymd"]).orderBy(F.col("proc_ym"),F.col("proc_ymd").desc()).show()

+-----+-----+-------+--------+----------+--------------+-----+---------+
|  idx|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|price_int|
+-----+-----+-------+--------+----------+--------------+-----+---------+
|86074|133.0| 202304|20230430|      상의|아바타파츠구분|  200|      200|
|67043|131.0| 202304|20230429|      헤어|아바타파츠구분|  150|      150|
|70507|135.0| 202304|20230428|    코스튬|아바타파츠구분|  300|      300|
|91146| 12.0| 202304|20230427|상태메시지|  기타파츠구분|   50|       50|
|42499| 13.0| 202304|20230426|상태메시지|  기타파츠구분|   50|       50|
|84515| 83.0| 202304|20230425|      신발|아바타파츠구분|  150|      150|
|46516|168.0| 202304|20230424|      헤어|아바타파츠구분|  150|      150|
| 4804|164.0| 202304|20230423|      신발|아바타파츠구분|   60|       60|
|89204| 60.0| 202304|20230422|      상의|아바타파츠구분|  200|      200|
|77389|118.0| 202304|20230421|  액세서리|아바타파츠구분|   25|       25|
|12389|167.0| 202304|20230420|      헤어|아바타파츠구분|  150|      150|
|83403|109.0| 202304|20230419|      신발|아바타파츠구분|   60|       60|
|87383| 39.0| 202304

### 9. Join & Union

#### 1) Join

In [170]:
#union은 잘 안쓰임
df1 = spark.read.csv("./test1.csv",header=True)
df3 = spark.read.csv("./test3.csv",header=True)
df3 = df3.withColumn("new_Name",F.col("Name"))
df1.show()
df3.show()

+-----+---+----------+------+
| Name|age|Experience|Salary|
+-----+---+----------+------+
|  Tom| 31|        10| 30000|
| Anne| 30|         8| 25000|
|Sunny| 29|         4| 20000|
| Paul| 24|         3| 20000|
| Mark| 21|         1| 15000|
|Jones| 23|         2| 18000|
+-----+---+----------+------+

+-----+------------+------+--------+
| Name| Departments|salary|new_Name|
+-----+------------+------+--------+
|  Tom|Data Science| 10000|     Tom|
|  Tom|         IOT|  5000|     Tom|
| Emma|    Big Data|  4000|    Emma|
|  Tom|    Big Data|  4000|     Tom|
| Emma|Data Science|  3000|    Emma|
| Mark|Data Science| 20000|    Mark|
| Mark|         IOT| 10000|    Mark|
| Mark|    Big Data|  5000|    Mark|
|Sunny|Data Science| 10000|   Sunny|
|Sunny|    Big Data|  2000|   Sunny|
+-----+------------+------+--------+



In [171]:
# join -> 테이블1.join(테이블2, join컬럼, join조건) 디폴트는 inner

df1.join(df3, df1.Name== df3.Name,"inner").show()

+-----+---+----------+------+-----+------------+------+--------+
| Name|age|Experience|Salary| Name| Departments|salary|new_Name|
+-----+---+----------+------+-----+------------+------+--------+
|  Tom| 31|        10| 30000|  Tom|Data Science| 10000|     Tom|
|  Tom| 31|        10| 30000|  Tom|         IOT|  5000|     Tom|
|  Tom| 31|        10| 30000|  Tom|    Big Data|  4000|     Tom|
| Mark| 21|         1| 15000| Mark|Data Science| 20000|    Mark|
| Mark| 21|         1| 15000| Mark|         IOT| 10000|    Mark|
| Mark| 21|         1| 15000| Mark|    Big Data|  5000|    Mark|
|Sunny| 29|         4| 20000|Sunny|Data Science| 10000|   Sunny|
|Sunny| 29|         4| 20000|Sunny|    Big Data|  2000|   Sunny|
+-----+---+----------+------+-----+------------+------+--------+



In [172]:
#join대상 컬럼의 이름이 다를 때
df1.join(df3, df1.Name == df3.new_Name,"inner").show()

+-----+---+----------+------+-----+------------+------+--------+
| Name|age|Experience|Salary| Name| Departments|salary|new_Name|
+-----+---+----------+------+-----+------------+------+--------+
|  Tom| 31|        10| 30000|  Tom|Data Science| 10000|     Tom|
|  Tom| 31|        10| 30000|  Tom|         IOT|  5000|     Tom|
|  Tom| 31|        10| 30000|  Tom|    Big Data|  4000|     Tom|
| Mark| 21|         1| 15000| Mark|Data Science| 20000|    Mark|
| Mark| 21|         1| 15000| Mark|         IOT| 10000|    Mark|
| Mark| 21|         1| 15000| Mark|    Big Data|  5000|    Mark|
|Sunny| 29|         4| 20000|Sunny|Data Science| 10000|   Sunny|
|Sunny| 29|         4| 20000|Sunny|    Big Data|  2000|   Sunny|
+-----+---+----------+------+-----+------------+------+--------+



In [173]:
#join 대상 컬럼의 이름이 같을 때 + 중복 제거
df1.join(df3, ['Name']).show() #->이러면 동일한 컬럼중에 하나만 남고 제거됨.

+-----+---+----------+------+------------+------+--------+
| Name|age|Experience|Salary| Departments|salary|new_Name|
+-----+---+----------+------+------------+------+--------+
|  Tom| 31|        10| 30000|Data Science| 10000|     Tom|
|  Tom| 31|        10| 30000|         IOT|  5000|     Tom|
|  Tom| 31|        10| 30000|    Big Data|  4000|     Tom|
| Mark| 21|         1| 15000|Data Science| 20000|    Mark|
| Mark| 21|         1| 15000|         IOT| 10000|    Mark|
| Mark| 21|         1| 15000|    Big Data|  5000|    Mark|
|Sunny| 29|         4| 20000|Data Science| 10000|   Sunny|
|Sunny| 29|         4| 20000|    Big Data|  2000|   Sunny|
+-----+---+----------+------+------------+------+--------+



In [174]:
# join 대상 컬럼이 여러개일때, 컬럼 이름이 다를 때
df1.join(df3, (df1.Name== df3.Name) & (df1.Salary == df3.salary)).show() #salary 값이 동일한 값이 없어서 아무것도 안나옴

+----+---+----------+------+----+-----------+------+--------+
|Name|age|Experience|Salary|Name|Departments|salary|new_Name|
+----+---+----------+------+----+-----------+------+--------+
+----+---+----------+------+----+-----------+------+--------+



In [175]:
df1 = df1.withColumnRenamed("Salary","salary")
df1.columns

['Name', 'age', 'Experience', 'salary']

In [176]:
df1.columns, df3.columns

(['Name', 'age', 'Experience', 'salary'],
 ['Name', 'Departments', 'salary', 'new_Name'])

In [177]:
# join 대상 컬럼이 여러개일때, 컬럼 이름이 같를 때
df1.join(df3, ['Name','salary']).show()

+----+------+---+----------+-----------+--------+
|Name|salary|age|Experience|Departments|new_Name|
+----+------+---+----------+-----------+--------+
+----+------+---+----------+-----------+--------+



#### 2) UNION
- union()
-unionByName()

In [178]:
#union() -> 컬럼 스키마(metadata)가 동일한 두 테이블에 대해 적용 가능 형샹이 같을 때만

In [179]:
#tes1 + test2
df2 = spark.read.csv("./test2.csv",header=True)

In [180]:
df1.count(), df2.count()

(6, 9)

In [181]:
df1.union(df2).show()

+-----+----+----------+------+
| Name| age|Experience|salary|
+-----+----+----------+------+
|  Tom|  31|        10| 30000|
| Anne|  30|         8| 25000|
|Sunny|  29|         4| 20000|
| Paul|  24|         3| 20000|
| Mark|  21|         1| 15000|
|Jones|  23|         2| 18000|
|  Tom|  31|        10| 30000|
| Anne|  30|         8| 25000|
|Sunny|  29|         4| 20000|
| Paul|  24|         3| 20000|
| Mark|  21|         1| 15000|
|Jones|  23|         2| 18000|
| Emma|NULL|      NULL| 40000|
| NULL|  34|        10| 38000|
| NULL|  36|      NULL|  NULL|
+-----+----+----------+------+



In [182]:
# unionByName() -> 컬럼이다른 테이블엗 union 적용이 가능

df1.unionByName(df3,allowMissingColumns=True).show() #공통되지 않은 부분도 union하겠다는 의미

+-----+----+----------+------+------------+--------+
| Name| age|Experience|salary| Departments|new_Name|
+-----+----+----------+------+------------+--------+
|  Tom|  31|        10| 30000|        NULL|    NULL|
| Anne|  30|         8| 25000|        NULL|    NULL|
|Sunny|  29|         4| 20000|        NULL|    NULL|
| Paul|  24|         3| 20000|        NULL|    NULL|
| Mark|  21|         1| 15000|        NULL|    NULL|
|Jones|  23|         2| 18000|        NULL|    NULL|
|  Tom|NULL|      NULL| 10000|Data Science|     Tom|
|  Tom|NULL|      NULL|  5000|         IOT|     Tom|
| Emma|NULL|      NULL|  4000|    Big Data|    Emma|
|  Tom|NULL|      NULL|  4000|    Big Data|     Tom|
| Emma|NULL|      NULL|  3000|Data Science|    Emma|
| Mark|NULL|      NULL| 20000|Data Science|    Mark|
| Mark|NULL|      NULL| 10000|         IOT|    Mark|
| Mark|NULL|      NULL|  5000|    Big Data|    Mark|
|Sunny|NULL|      NULL| 10000|Data Science|   Sunny|
|Sunny|NULL|      NULL|  2000|    Big Data|   

### 10. Window
<img src="https://velog.velcdn.com/images/newnew_daddy/post/a5c7efcc-6f4b-49d0-970f-a9fefdeec23f/image.png" width="50%">    
<img src="https://velog.velcdn.com/images/newnew_daddy/post/a5ffc29c-5e9c-40fb-a749-63f7694f43d3/image.png" width="50%">

> https://sparkbyexamples.com/pyspark/pyspark-window-functions/

In [184]:
df3= df3.drop("new_Name")
df3.show()


+-----+------------+------+
| Name| Departments|salary|
+-----+------------+------+
|  Tom|Data Science| 10000|
|  Tom|         IOT|  5000|
| Emma|    Big Data|  4000|
|  Tom|    Big Data|  4000|
| Emma|Data Science|  3000|
| Mark|Data Science| 20000|
| Mark|         IOT| 10000|
| Mark|    Big Data|  5000|
|Sunny|Data Science| 10000|
|Sunny|    Big Data|  2000|
+-----+------------+------+



In [185]:
df3 = df3.withColumn("salary", F.col("salary").cast(T.IntegerType()))
df3.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Departments: string (nullable = true)
 |-- salary: integer (nullable = true)



In [187]:
from pyspark.sql import window as w

window_var =w.Window.partitionBy(F.col("Name")).orderBy(F.col("salary").desc())
df3.withColumn("salary_rank", F.row_number().over(window_var)).filter(F.col("salary_rank")==1).show()

+-----+------------+------+-----------+
| Name| Departments|salary|salary_rank|
+-----+------------+------+-----------+
| Emma|    Big Data|  4000|          1|
| Mark|Data Science| 20000|          1|
|Sunny|Data Science| 10000|          1|
|  Tom|Data Science| 10000|          1|
+-----+------------+------+-----------+



### 11. UDF(User Defined Function)

> https://velog.io/@newnew_daddy/spark05

In [ ]:
# 1) udf에 직접 등록

In [189]:
## Departments 컬럼 -> 공백에는 '@'를 채워주고, 모두 대문자로 변형(Big Data ->BIG@DATA)

var = "Data Science"

def str_udf(var):
  #대문자로 변경
  var = var.upper()
  #공백을@로 치환
  var = var.replace(" ", "@")
  return var
str_udf(var)

'DATA@SCIENCE'

In [190]:
# udf에 등록
user_udf = F.udf(str_udf,returnType = T.StringType())
df3.withColumn("new_Departments", user_udf(F.col("Departments"))).show()

+-----+------------+------+---------------+
| Name| Departments|salary|new_Departments|
+-----+------------+------+---------------+
|  Tom|Data Science| 10000|   DATA@SCIENCE|
|  Tom|         IOT|  5000|            IOT|
| Emma|    Big Data|  4000|       BIG@DATA|
|  Tom|    Big Data|  4000|       BIG@DATA|
| Emma|Data Science|  3000|   DATA@SCIENCE|
| Mark|Data Science| 20000|   DATA@SCIENCE|
| Mark|         IOT| 10000|            IOT|
| Mark|    Big Data|  5000|       BIG@DATA|
|Sunny|Data Science| 10000|   DATA@SCIENCE|
|Sunny|    Big Data|  2000|       BIG@DATA|
+-----+------------+------+---------------+



In [ ]:
# 2) udf decorator를 사용

In [200]:
@F.udf(returnType= T.IntegerType()) #returnType default가 스트링타입 그래서 스트링타입일 경우 생략 가능.
def str_udf(var):
  var = var.upper()
  var = var.replace(" ", "@")
  return var

In [201]:
df3.withColumn("new_Departments", str_udf(F.col("Departments"))).show()

+-----+------------+------+---------------+
| Name| Departments|salary|new_Departments|
+-----+------------+------+---------------+
|  Tom|Data Science| 10000|           NULL|
|  Tom|         IOT|  5000|           NULL|
| Emma|    Big Data|  4000|           NULL|
|  Tom|    Big Data|  4000|           NULL|
| Emma|Data Science|  3000|           NULL|
| Mark|Data Science| 20000|           NULL|
| Mark|         IOT| 10000|           NULL|
| Mark|    Big Data|  5000|           NULL|
|Sunny|Data Science| 10000|           NULL|
|Sunny|    Big Data|  2000|           NULL|
+-----+------------+------+---------------+



### 12. 기타


##### 1) 컬럼 이름 일괄 변환

In [202]:
# df3.withColumnRenamed().withColumnRenamed().withColumnRenamed()
chage_col=["Name_a","Department_a","salary_a"]
df3.toDF(*chage_col).show()

+------+------------+--------+
|Name_a|Department_a|salary_a|
+------+------------+--------+
|   Tom|Data Science|   10000|
|   Tom|         IOT|    5000|
|  Emma|    Big Data|    4000|
|   Tom|    Big Data|    4000|
|  Emma|Data Science|    3000|
|  Mark|Data Science|   20000|
|  Mark|         IOT|   10000|
|  Mark|    Big Data|    5000|
| Sunny|Data Science|   10000|
| Sunny|    Big Data|    2000|
+------+------------+--------+



##### 2) 단일값의 컬럼 추가

In [203]:
# 테이블이 여러개 일때, 해당 join이 일어난 후 특정 컬럼이 어떤 테이블에서 왔는지 확인하고자 하는 목적
# 데이터 소스를 확인하기 쉬움

df3.withColumn("kdt",F.lit("7")).show()

+-----+------------+------+---+
| Name| Departments|salary|kdt|
+-----+------------+------+---+
|  Tom|Data Science| 10000|  7|
|  Tom|         IOT|  5000|  7|
| Emma|    Big Data|  4000|  7|
|  Tom|    Big Data|  4000|  7|
| Emma|Data Science|  3000|  7|
| Mark|Data Science| 20000|  7|
| Mark|         IOT| 10000|  7|
| Mark|    Big Data|  5000|  7|
|Sunny|Data Science| 10000|  7|
|Sunny|    Big Data|  2000|  7|
+-----+------------+------+---+



##### 3) 특정 문자를 기준으로 split

In [206]:
df3.withColumns("split_col", F.split(F.col("Departments")," ").getItem(0)).show() #split_col에서 나눠진 컬럼의 첫번째 항을 가져옴(getItem)

AssertionError: 

##### 4) spark dataframe  <-->  pandas dataframe

In [208]:
# 1) spark dataframe -. pandas dataframe
pdf3 = df3.toPandas()
type(pdf3)

pandas.core.frame.DataFrame

In [210]:
pdf3.head(), len(pdf3)

(   Name   Departments  salary
 0   Tom  Data Science   10000
 1   Tom           IOT    5000
 2  Emma      Big Data    4000
 3   Tom      Big Data    4000
 4  Emma  Data Science    3000,
 10)

In [211]:
# 2) pandas dataframe -> spark dataframe

sdf3 = spark.createDataFrame(pdf3)
type(sdf3)

pyspark.sql.dataframe.DataFrame

In [214]:
sdf3.show()
sdf3.head()

+-----+------------+------+
| Name| Departments|salary|
+-----+------------+------+
|  Tom|Data Science| 10000|
|  Tom|         IOT|  5000|
| Emma|    Big Data|  4000|
|  Tom|    Big Data|  4000|
| Emma|Data Science|  3000|
| Mark|Data Science| 20000|
| Mark|         IOT| 10000|
| Mark|    Big Data|  5000|
|Sunny|Data Science| 10000|
|Sunny|    Big Data|  2000|
+-----+------------+------+



Row(Name='Tom', Departments='Data Science', salary=10000)